In [1]:
# =============================================================
# Teste do agente de monitoramento
# Pipeline com triagem, deteccao da CNN, planejamento, resumo e ordem tatica.
# =============================================================
import os
import time

# Modelos locais maiores (ex.: qwen35-9b) sao lentos em CPU; ampliamos o timeout
# por chamada de LLM ANTES de importar agent.settings (que le a env var no import).
os.environ["LLM_TIMEOUT"] = "300"

from agent.settings import STATIONS_DIR
from agent.vision import classify_image
from agent.monitoring_flow import (
    make_llm, build_recursos, _build_report,
    triar_mensagem, _build_triage_agent,
    _build_planner_agent, _build_alert_agent, _build_navegador_agent,
)
from agent.situation import (
    QUADRO, FROTA, resumo_md, frota_md,
    _alocar_demandas, planejar_demandas, despachar_alocacao,
    gerar_briefing, briefing_da_missao,
)

# Servidor LLM dos agentes (LiteLLM)
PROVIDER = "ollama" # ollama, openai, anthropic, azure, gemini
MODEL = "text-qwen35-9b" # modelo local; liste os disponiveis com "ollama list"
SERVER_URL = "http://localhost:11434"
API_KEY = ""

llm = make_llm(provider=PROVIDER, model=MODEL,
               base_url=SERVER_URL, api_key=API_KEY or None)
QUADRO.limpar()
FROTA.limpar()

# Tempo de processamento (s) por agente, preenchido ao longo do notebook.
latencias = {}

In [2]:
# =============================================================
# Triagem das mensagens de vitimas
# =============================================================
mensagens = [
    "socorro tem 4 pessoas presas no telhado da rua das flores 120, a agua ta subindo rapido e tem uma criança pequena",
    "meu pai e cadeirante e a agua ja ta na altura do peito aqui na vila são jose, preciso de resgate urgente agora",
    "minha vó ta sozinha no bairro navegantes e a rua alagou toda, ela nao consegue sair de casa",
    "somos 8 pessoas no abrigo da escola municipal, sem agua potavel nem comida ha 2 dias",
    "alguem sabe se o mercado do centro abriu? queria comprar pão",
]

triador = _build_triage_agent(llm)
tempos = []
for m in mensagens:
    t0 = time.perf_counter()
    r = triar_mensagem(m, triador)
    tempos.append(time.perf_counter() - t0)
    print(m)
    if r is None:
        print("  triagem indisponivel (sem LLM)\n")
        continue
    print(f"  local={r.local} | pessoas={r.pessoas} | necessidade={r.necessidade} | urgencia={r.urgencia}")
    print(f"  resumo: {r.resumo}\n")
    QUADRO.publicar_vitima(r)

latencias["Triagem"] = sum(tempos) / len(tempos)
print(f"Latencia media da triagem: {latencias['Triagem']:.1f} s por mensagem ({len(mensagens)} mensagens)")

socorro tem 4 pessoas presas no telhado da rua das flores 120, a agua ta subindo rapido e tem uma criança pequena
  local=Rua das Flores, 120 | pessoas=4 | necessidade=resgate | urgencia=critica
  resumo: 4 pessoas presas no telhado, água subindo rápido, presença de criança pequena



meu pai e cadeirante e a agua ja ta na altura do peito aqui na vila são jose, preciso de resgate urgente agora
  local=Vila São José | pessoas=2 | necessidade=resgate | urgencia=critica
  resumo: Remetente e pai (cadeirante)



minha vó ta sozinha no bairro navegantes e a rua alagou toda, ela nao consegue sair de casa
  local=Bairro Navegantes | pessoas=1 | necessidade=resgate | urgencia=critica
  resumo: Vó idosa sozinha, presa na residência devido ao alagamento total da rua.



somos 8 pessoas no abrigo da escola municipal, sem agua potavel nem comida ha 2 dias
  local=abrigo da escola municipal | pessoas=8 | necessidade=mantimentos | urgencia=alta
  resumo: Grupo de 8 pessoas em abrigo escolar sem água potável e comida há 2 dias.



alguem sabe se o mercado do centro abriu? queria comprar pão
  local=Mercado do Centro | pessoas=0 | necessidade=mantimentos | urgencia=baixa
  resumo: Consulta sobre abertura do comércio para aquisição de alimentos (pão)

Latencia media da triagem: 54.1 s por mensagem (5 mensagens)


In [3]:
# =============================================================
# Drone: a CNN classifica imagens da estacao e publica eventos
# =============================================================
estacao = "Centro_Historico"
pasta = os.path.join(STATIONS_DIR, estacao)
for nome in sorted(os.listdir(pasta)):
    res = classify_image(os.path.join(pasta, nome))
    rep = _build_report(estacao.replace("_", " "), [res])
    print(f"{nome}: prob={res.probability:.1%} | enchente={res.flooded} | severidade={rep.severity}")
    if res.flooded:
        QUADRO.publicar_deteccao(rep.station, res.probability, rep.severity)

1005_1.png: prob=73.5% | enchente=True | severidade=medio
12002_1.png: prob=86.2% | enchente=True | severidade=alto
13008.png: prob=12.4% | enchente=False | severidade=normal
3008_1.png: prob=45.3% | enchente=False | severidade=baixo
6007_1.png: prob=93.5% | enchente=True | severidade=alto


In [4]:
# =============================================================
# Quadro de situacao consolidado (drones + triagem)
# =============================================================
print(resumo_md(QUADRO))

### Quadro de situação
- Eventos de enchente (drones): **3**
- Pedidos de vítima (triagem): **5**
- Demandas ativas (priorizadas): **5**

| Prioridade | Tipo | Local | Detalhe |
|:--:|---|---|---|
| 4 | vitima | Rua das Flores, 120 | 4 pessoa(s) - resgate |
| 4 | vitima | Vila São José | 2 pessoa(s) - resgate |
| 4 | vitima | Bairro Navegantes | 1 pessoa(s) - resgate |
| 3 | vitima | abrigo da escola municipal | 8 pessoa(s) - mantimentos |
| 3 | enchente | Centro Historico | enchente alto (prob 93%) |


In [5]:
# =============================================================
# Planejador: aloca a frota sobre as demandas e despacha
# =============================================================
FROTA.definir_total(build_recursos(Helicóptero=1, Bote=2, Equipe_terrestre=3))
demandas = QUADRO.demandas()
aloc = _alocar_demandas(demandas, FROTA.disponiveis_lista())
t0 = time.perf_counter()
plano = planejar_demandas(aloc, _build_planner_agent(llm))
latencias["Planejador"] = time.perf_counter() - t0
despachar_alocacao(aloc, FROTA)
print(plano)
print(f"\n(Latencia do planejador: {latencias['Planejador']:.1f} s)")

1. Ordem de Prioridade:
- Prioridade 4: Rua das Flores, 120 (Helicóptero + Bote), Vila São José (Bote + Equipe terrestre), Bairro Navegantes (Equipe terrestre + Equipe terrestre).
- Prioridade 3: Abrigo da escola municipal e Centro Historico.
2. Justificativas:
- Rua das Flores, 120: Helicóptero e bote alocados para resgate de 4 pessoas.
- Vila São José: Bote e equipe terrestre alocados para resgate de 2 pessoas.
- Bairro Navegantes: Duas equipes terrestres alocadas para resgate de 1 pessoa.
3. Lacunas:
- Abrigo da escola municipal (vitima, prioridade 3) e Centro Historico (enchente, prioridade 3): SEM RECURSO DISPONÍVEL.

(Latencia do planejador: 131.3 s)


In [6]:
# =============================================================
# Monitoramento: resumo da situacao
# =============================================================
t0 = time.perf_counter()
resumo = gerar_briefing(QUADRO.demandas(), _build_alert_agent(llm))
latencias["Monitoramento"] = time.perf_counter() - t0
print(resumo)
print(f"\n(Latencia do monitoramento: {latencias['Monitoramento']:.1f} s)")

Situação consolidada com 5 demandas ativas, distribuídas entre prioridade 3 e 4. Pontos críticos: Centro Histórico (enchente alta, 93% de probabilidade) e Rua das Flores, 120 (4 pessoas para resgate). Também requer atenção o abrigo da escola municipal (8 pessoas - mantimentos) e as demais áreas de resgate (Vila São José e Navegantes). Alerta: Vítimas em risco iminente nas localizações de resgate e impacto estrutural no Centro Histórico.

(Latencia do monitoramento: 114.8 s)


In [7]:
# =============================================================
# Navegador: ordem tatica de uma missao despachada
# =============================================================
m = FROTA.ativas[0]
print(f"{m.recurso} -> {m.local} ({m.tipo}, prioridade {m.prioridade})")
t0 = time.perf_counter()
ordem = briefing_da_missao(m, _build_navegador_agent(llm))
latencias["Navegador"] = time.perf_counter() - t0
print(ordem)
print(f"\n(Latencia do navegador: {latencias['Navegador']:.1f} s)")

Helicóptero -> Rua das Flores, 120 (vitima, prioridade 4)


Deslocar-se imediatamente para Rua das Flores, 120 com prioridade máxima.
Ao chegar, identificar as 4 pessoas e preparar área de pouso segura.
Estabilizar as vítimas e executar a extração prioritária com foco na segurança da equipe.

(Latencia do navegador: 85.7 s)


In [8]:
# =============================================================
# Estado final da frota
# =============================================================
print(frota_md(FROTA))

### Recursos e equipes
| Recurso | Total | Ocupadas | Livres |
|---|:--:|:--:|:--:|
| Helicóptero | 1 | 1 | 0 |
| Bote | 2 | 2 | 0 |
| Equipe terrestre | 3 | 3 | 0 |

**Missões ativas:**
| Recurso | Destino | Tipo | Despachada em |
|---|---|---|---|
| Helicóptero | Rua das Flores, 120 | vitima | 2026-06-10T18:03:53 |
| Bote | Rua das Flores, 120 | vitima | 2026-06-10T18:03:53 |
| Bote | Vila São José | vitima | 2026-06-10T18:03:53 |
| Equipe terrestre | Vila São José | vitima | 2026-06-10T18:03:53 |
| Equipe terrestre | Bairro Navegantes | vitima | 2026-06-10T18:03:53 |
| Equipe terrestre | Bairro Navegantes | vitima | 2026-06-10T18:03:53 |


In [9]:
# =============================================================
# Tempo de processamento: latencia por agente (chamada de LLM)
# =============================================================
print(f"Tempo por chamada de agente (modelo: {MODEL})
")
print(f"{'Agente':<16}{'Tempo (s)':>10}")
print("-" * 26)
for nome, seg in latencias.items():
    print(f"{nome:<16}{seg:>10.1f}")
print("
A triagem e a media das 5 mensagens. Os outros sao 1 chamada cada.")
print("Medido em uma RTX 3070 de 8 GB. O modelo de 9B pensa antes de responder, por isso a demora.")

Tempo por chamada de agente (modelo: text-qwen35-9b)

Agente           Tempo (s)
--------------------------
Triagem               54.1
Planejador           131.3
Monitoramento        114.8
Navegador             85.7

A triagem e a media das 5 mensagens. Os outros sao 1 chamada cada.
Medido em uma RTX 3070 de 8 GB. O modelo de 9B pensa antes de responder, por isso a demora.


## Conclusões dos testes

Rodei o fluxo inteiro com os quatro agentes e eles funcionaram juntos: a triagem leu as mensagens, a CNN classificou as imagens da estação, o quadro de situação juntou as duas fontes, o planejador distribuiu a frota e o navegador escreveu a ordem de cada equipe.

O que observei nos testes:

- A triagem se saiu bem com mensagens escritas de qualquer jeito, com gírias e erros, e ainda separou o que não era emergência. A mensagem perguntando se o mercado abriu ficou com urgência baixa e não entrou na lista de demandas.
- A CNN achou 3 enchentes nas 5 imagens da estação, um resultado parecido com a acurácia que medi na Parte 1, perto de 0,85.
- O quadro juntou os eventos dos drones com os pedidos das vítimas e montou 5 demandas já em ordem de prioridade. Quando dá empate, a vítima fica na frente da enchente.
- O planejador respeitou o que tinha de frota e avisou onde faltou recurso, marcando as demandas como "SEM RECURSO DISPONÍVEL".
- O navegador devolveu uma ordem curta e direta para a unidade.

Sobre o tempo de processamento, cada chamada de LLM levou de um a pouco mais de dois minutos com o Qwen3.5 9B, como mostra a tabela acima, bem acima dos 40 ms da CNN por imagem. No começo achei que fosse a VRAM, porque rodei numa RTX 3070 de 8 GB, mas olhando o ollama show descobri o motivo real. O modelo de 9B é um modelo de raciocínio, ele gera um bloco de pensamento antes da resposta final. Esse pensamento gasta muitos tokens e tempo, e o Ollama devolve esse bloco separado, então a resposta que aparece fica limpa mas a chamada demora. O Qwen3 4B que testei antes não faz isso, ele já vem com o raciocínio desligado, e por isso responde em 2 a 3 segundos.

Ou seja, a diferença de tempo entre os dois modelos vem mais do raciocínio do que do tamanho ou da placa de vídeo. Mesmo com a VRAM livre o 9B continuou lento. Isso reforça a divisão do projeto: a CNN, que é rápida, fica na borda no drone, e os agentes de LLM, mais lentos, ficam no centro de comando e só são chamados quando chega um evento.

Vale lembrar que esse teste é mais qualitativo, ou seja, olhei se a saída faz sentido, diferente da CNN que tem uma acurácia medida. E o quadro e a frota ficam guardados na memória do programa, então a demonstração roda em uma sessão só.